# 3. Build an SVD from blocks and recipes

`svdlab.blocks` are the steps of SVD algorithms (QR, bidiagonalization, bidiagonal SVD,
Jacobi, ...). `svdlab.recipes` are complete SVDs built from them, where every step is a
keyword argument you can replace. Text version: [tutorial 3](../docs/tutorials/03-blocks-and-recipes.md).
About two minutes on a laptop CPU.

```text
recipes.golub_kahan   a ──bidiagonalize──► Q, B, P ──bidiagonal_svd──► U = Q U_B, V = P V_B
recipes.qr_then_svd   a ──qr──► Q, R ──small_svd(R)──► U = Q U_R
recipes.randomized    a ──sketch──► A Ω ──qr──► Q ──(power iterations: qr)──► small_svd(Qᴴ A) ──► U = Q U_small
```

In [1]:
# Setup: find the repository, choose the device. Run this cell first.
from pathlib import Path
import os, sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'svdlab' / '__init__.py').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))      # works even if the package is not installed with pip
os.chdir(ROOT)                         # all paths below are relative to the repository root

import torch
import svdlab

USE_GPU = True                         # set to False to stay on the CPU even when a GPU is present
DEVICE = 'cuda' if USE_GPU and torch.cuda.is_available() else 'cpu'
torch.set_num_threads(4)
print(f'torch {torch.__version__}, device: {DEVICE}' + (f' ({torch.cuda.get_device_name(0)})' if DEVICE == 'cuda' else ''))

torch 2.14.1+cu130, device: cpu


## Blocks, one at a time

The blocks are the repository's own implementations (tensor arithmetic only), so they run
on CPU and GPU, real and complex. Tall-only blocks (`qr`, `bidiagonalize`) reject wide
matrices; the recipes pass them the adjoint instead.

In [2]:
from svdlab import blocks, recipes

a = torch.randn(8, 5, dtype=torch.float64, device=DEVICE)

q, r = blocks.qr(a)
print('QR:            |QR - A| =', float(torch.linalg.norm(q @ r - a)), ' R upper:', bool(torch.allclose(r, r.triu())))

q, d, e, p = blocks.bidiagonalize(a)          # A = Q B Pᴴ, B upper bidiagonal (d, e)
b = torch.diag(d) + torch.diag(e, 1)
print('bidiagonalize: |Q B Pᴴ - A| =', float(torch.linalg.norm(q @ b @ p.mH - a)))

ub, s, vb = blocks.bidiagonal_svd(d, e)       # B = Ub diag(s) Vbᵀ  (note: V, not Vh)
print('bidiagonal_svd: s =', s.cpu().numpy().round(4))
print('torch says:        ', torch.linalg.svdvals(a).cpu().numpy().round(4))

QR:            |QR - A| = 1.2629168157263127e-15  R upper: True
bidiagonalize: |Q B Pᴴ - A| = 1.320232838524727e-15
bidiagonal_svd: s = [3.677  2.7614 2.0229 1.4046 0.8086]
torch says:         [3.677  2.7614 2.0229 1.4046 0.8086]


In [3]:
print('All blocks:', ', '.join(n for n in blocks.__all__ if n != 'ALGORITHMS'))
print('Recipes:   ', ', '.join(recipes.__all__))

All blocks: qr, torch_qr, bidiagonalize, bidiagonal_svd, jacobi_svd, tridiagonalize, hermitian_eig, cholesky, solve_lower, solve_upper, random_sketch, complete_basis, svd, torch_svd
Recipes:    golub_kahan, qr_then_svd, randomized, jacobi


## The recipes as they are

Each recipe has the `(a, rank)` signature, so it can be checked directly.

In [4]:
rows = svdlab.compare({'golub_kahan': recipes.golub_kahan, 'qr_then_svd': recipes.qr_then_svd,
                       'randomized': recipes.randomized}, svdlab.QUICK_SUITE, device=DEVICE, verbose=False)
svdlab.table(rows).pivot(index='case', columns='method', values='ok')

method,golub_kahan,qr_then_svd,randomized
case,,,
clustered,True,True,True
complex_gaussian,True,True,True
gaussian_tall,True,True,True
gaussian_wide,True,True,True
ill_conditioned_1e8,True,True,True
low_rank_k4,True,True,True


## Replace a step: CholeskyQR2 inside randomized SVD

CholeskyQR2 computes a QR from the Cholesky factor of YᴴY. It is two matrix products and
two small triangular operations, which makes it fast on GPUs. It must keep the contract of
`blocks.qr`: tall `y (m, n)` in, `Q (m, n)` and upper `R (n, n)` out.

In [5]:
def cholesky_qr2(y):
    """Same contract as svdlab.blocks.qr: tall y (m, n) -> Q (m, n) orthonormal, R (n, n) upper."""
    q, r = y, torch.eye(y.shape[1], dtype=y.dtype, device=y.device)
    for _ in range(2):                            # the second pass repairs the lost orthogonality
        l = torch.linalg.cholesky(q.mH @ q)       # QᴴQ = L Lᴴ
        q = torch.linalg.solve_triangular(l.mH, q, upper=True, left=False)   # Q L⁻ᴴ
        r = l.mH @ r
    return q, r

def randomized_cholqr2(a, rank):
    return recipes.randomized(a, rank, qr=cholesky_qr2)

# The quick matrices plus the larger ill-conditioned one from the default set.
hard = [c for c in svdlab.DEFAULT_SUITE if c['name'] == 'ill_conditioned_1e8']
cases = list(svdlab.QUICK_SUITE) + [dict(hard[0], name='ill_conditioned_1e8_160x120')]
rows = svdlab.compare({'randomized': recipes.randomized, 'randomized_cholqr2': randomized_cholqr2},
                      cases, device=DEVICE, repeat=3)

randomized           gaussian_tall          ok      83.504 ms  recon 5.02e-15  tail× -  σ-rel 4.32e-15  orth 5.37e-15


randomized           gaussian_wide          ok      94.860 ms  recon 4.73e-15  tail× -  σ-rel 4.29e-15  orth 5.01e-15


randomized           ill_conditioned_1e8    ok     135.145 ms  recon 8.58e-15  tail× -  σ-rel 3.00e-10  orth 8.84e-15


randomized           clustered              ok      90.164 ms  recon 1.07e-14  tail× -  σ-rel 2.17e-14  orth 1.00e-14


randomized           complex_gaussian       ok     104.582 ms  recon 2.91e-15  tail× -  σ-rel 2.70e-15  orth 3.14e-15


...

randomized_cholqr2   gaussian_wide          ok      65.599 ms  recon 4.80e-15  tail× -  σ-rel 4.17e-15  orth 4.88e-15


randomized_cholqr2   ill_conditioned_1e8    ok     156.096 ms  recon 8.11e-15  tail× -  σ-rel 4.48e-10  orth 9.20e-15


randomized_cholqr2   clustered              ok      79.039 ms  recon 1.16e-14  tail× -  σ-rel 1.82e-14  orth 1.00e-14


randomized_cholqr2   complex_gaussian       ok      83.580 ms  recon 3.40e-15  tail× -  σ-rel 3.08e-15  orth 3.40e-15


randomized_cholqr2   low_rank_k4            ok      14.015 ms  recon 6.64e-06  tail× 1.00e+00  σ-rel 5.55e-16  orth 1.16e-15
randomized_cholqr2   ill_conditioned_1e8_160x120 ERROR _LinAlgError: linalg.cholesky: The factorization could not be completed because the input is not positive-definite (the leading minor of order 111 is not positive-definite).


CholeskyQR2 is faster and fails on the larger ill-conditioned matrix: YᴴY squares the
condition number, and with a condition number of 1e8 the Gram matrix is no longer positive
definite in float64. The failing case is reported as `ERROR` with the exception message and
does not stop the other cases. The same file from the terminal:

```bash
python -m svdlab check examples/replace_a_step.py --compare svdlab.recipes:randomized
```

## Your turn: your own QR

Below is classical Gram–Schmidt with re-orthogonalization (CGS2) as an example of a QR
written from scratch. Change it, or write a different one, and see where it breaks.

In [6]:
def my_qr(y):
    """YOUR CODE HERE. Classical Gram–Schmidt, twice per column (CGS2); same contract as blocks.qr."""
    m, n = y.shape
    q = torch.zeros_like(y)
    r = torch.zeros(n, n, dtype=y.dtype, device=y.device)
    for j in range(n):
        v = y[:, j]
        for _ in range(2):                        # the second pass removes what rounding left behind
            c = q[:, :j].mH @ v
            v = v - q[:, :j] @ c
            r[:j, j] += c
        r[j, j] = torch.linalg.vector_norm(v)
        q[:, j] = v / r[j, j]
    return q, r

rows = svdlab.check(lambda a, rank: recipes.randomized(a, rank, qr=my_qr), cases,
                    name='randomized_my_qr', device=DEVICE, repeat=1)

randomized_my_qr     gaussian_tall          ok      86.658 ms  recon 5.43e-15  tail× -  σ-rel 4.32e-15  orth 5.74e-15


randomized_my_qr     gaussian_wide          ok      51.449 ms  recon 4.81e-15  tail× -  σ-rel 3.83e-15  orth 4.94e-15


randomized_my_qr     ill_conditioned_1e8    ok     110.962 ms  recon 8.15e-15  tail× -  σ-rel 3.94e-10  orth 8.99e-15


randomized_my_qr     clustered              ok      99.210 ms  recon 1.11e-14  tail× -  σ-rel 1.20e-14  orth 1.07e-14


randomized_my_qr     complex_gaussian       ok      59.280 ms  recon 3.23e-15  tail× -  σ-rel 3.47e-15  orth 3.31e-15


randomized_my_qr     low_rank_k4            ok      26.888 ms  recon 6.64e-06  tail× 1.00e+00  σ-rel 4.78e-16  orth 9.83e-16


randomized_my_qr     ill_conditioned_1e8_160x120 ok    1993.346 ms  recon 4.77e-14  tail× -  σ-rel 4.44e-10  orth 5.19e-14


## Find the slow step

Swapping a block for the library version (`blocks.torch_qr`, `blocks.torch_svd`, or
`torch.linalg.svd` on B) shows where the time goes. `functools.partial(recipe, step=...)`
makes an `(a, rank)` function without writing a wrapper.

In [7]:
import functools

def library_bidiagonal_svd(d, e):
    """Same contract as blocks.bidiagonal_svd: B = U diag(s) Vᵀ, returns (U, s, V)."""
    u, s, vh = torch.linalg.svd(torch.diag(d) + torch.diag(e, 1))
    return u, s, vh.mT

methods = {
    'golub_kahan': recipes.golub_kahan,
    'golub_kahan, library bidiagonal_svd': functools.partial(recipes.golub_kahan, bidiagonal_svd=library_bidiagonal_svd),
    'qr_then_svd': recipes.qr_then_svd,
    'qr_then_svd, library qr and small_svd': functools.partial(recipes.qr_then_svd, qr=blocks.torch_qr,
                                                               small_svd=blocks.torch_svd),
}
rows = svdlab.compare(methods, svdlab.QUICK_SUITE, device=DEVICE, repeat=3, verbose=False)
for s in svdlab.summary(rows):
    print(f"{s['method']:<40} {s['ok']}/{s['cases']} ok   total {s['total_median_ms']:9.1f} ms")

golub_kahan                              6/6 ok   total    3854.7 ms
golub_kahan, library bidiagonal_svd      6/6 ok   total     102.0 ms
qr_then_svd                              6/6 ok   total     740.0 ms
qr_then_svd, library qr and small_svd    6/6 ok   total       2.3 ms


In Golub–Kahan almost all the time is in the bidiagonal SVD (`bdsqr`, a Python loop of
implicit-shift QR steps here): that is the step where a faster implementation pays off.

## Tune a parameter

`recipes.randomized` also takes `oversample` (default 8), `power_iterations` (default 2) and
`seed`. A truncated case passes when the residual is within 1.05× the optimal one (`tail×`).

In [8]:
cases = [
    dict(name='low_rank_k10', m=300, n=200, family='low_rank', rank=10, tail=1e-6, output_rank=10),
    dict(name='slow_decay', m=300, n=200, family='geometric', condition=1e3, output_rank=10),
]
variants = {
    'p0': functools.partial(recipes.randomized, power_iterations=0),
    'p0_oversample30': functools.partial(recipes.randomized, power_iterations=0, oversample=30),
    'p2 (default)': recipes.randomized,
}
rows = svdlab.compare(variants, cases, device=DEVICE, repeat=3, verbose=False)
svdlab.table(rows).pivot(index='method', columns='case', values='residual_over_optimal_tail').round(3)

case,low_rank_k10,slow_decay
method,,
p0,1.384,1.104
p0_oversample30,1.116,1.033
p2 (default),1.000,1.002


## Compose your own chain

A recipe is itself an SVD and can be a step of another recipe, and a new chain of blocks is
a few lines:

In [9]:
qr_then_golub_kahan = functools.partial(recipes.qr_then_svd, small_svd=recipes.golub_kahan)

def qr_bidiag_chain(a, rank):
    """a = Q R, R = Q2 B Pᴴ, B = Ub diag(s) Vbᵀ, written out step by step."""
    wide = a.shape[0] < a.shape[1]
    x = a.mH if wide else a
    q, r = blocks.qr(x)
    q2, d, e, p = blocks.bidiagonalize(r)
    ub, s, vb = blocks.bidiagonal_svd(d, e)
    u = q @ q2 @ ub.to(x.dtype)                   # ub, vb are real; make them complex for complex input
    v = p @ vb.to(x.dtype)
    u, s, vh = u[:, :rank], s[:rank], v[:, :rank].mH
    return (vh.mH, s, u.mH) if wide else (u, s, vh)

rows = svdlab.compare({'qr_then_golub_kahan': qr_then_golub_kahan, 'qr_bidiag_chain': qr_bidiag_chain},
                      svdlab.QUICK_SUITE, device=DEVICE, repeat=1, verbose=False)
for s in svdlab.summary(rows):
    print(f"{s['method']:<22} {s['ok']}/{s['cases']} ok")

qr_then_golub_kahan    6/6 ok
qr_bidiag_chain        6/6 ok


## Next

- [04_native_cpp_cuda.ipynb](04_native_cpp_cuda.ipynb): write a step or a whole SVD in C++ or CUDA.
- The 16 complete repository SVDs are available as builtin names (`'bidiag_qr'`,
  `'polar_qdwh'`, ...) and as `blocks.svd(a, algorithm)`.